# Aula 21: Árvores Binárias no Mapeamento e Tomada de Decisão do Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, implementaremos a estrutura de **Árvore Binária de Busca (BST - Binary Search Tree)** e árvores de decisão comportamental focadas no sistema **Visão-AGV**.

### Aplicações Práticas no AGV
1. **Fila de Prioridade Dinâmica de Tarefas (BST):** Organização de ordens de transporte de materiais indexadas por nível de urgência, permitindo inserção e busca em tempo $O(\log N)$.
2. **Decomposição e Consulta Espacial:** Representação da hierarquia de zonas operacionais da fábrica para rápida checagem de permissão de acesso e status de ocupação.
3. **Algoritmos de Travessia:** Implementação dos métodos Pré-Ordem, Em-Ordem e Pós-Ordem para auditoria e serialização do estado dos sensores e atuadores.

---

In [ ]:
# Estrutura de Nó de uma Árvore Binária para o Sistema Visão-AGV

class NoAGV:
    """
    Representa um nó na árvore binária de controle do AGV.
    Pode armazenar uma ordem de transporte (com prioridade) ou uma subzona da fábrica.
    """
    def __init__(self, chave_prioridade, id_tarefa, zona_destino):
        self.prioridade = chave_prioridade  # Chave numérica de prioridade (quanto menor, mais urgente)
        self.id_tarefa = id_tarefa          # Identificador da ordem (ex: 'ORD-102')
        self.zona_destino = zona_destino    # Estação ou zona de entrega (ex: 'R-101')
        self.esquerda = None                 # Subárvore esquerda (menores prioridades / mais urgentes)
        self.direita = None                  # Subárvore direita (maiores prioridades / menos urgentes)

    def __repr__(self):
        return f"[{self.id_tarefa} | Prio: {self.prioridade} -> {self.zona_destino}]"

print("Classe NoAGV declarada com sucesso!")

## 2. Implementação da Árvore Binária de Busca (BST)

A classe `ArvoreDecisaoAGV` gerencia as operações de inserção, busca e travessia das tarefas do robô autônomo.

In [ ]:
class ArvoreDecisaoAGV:
    def __init__(self):
        self.raiz = None

    def inserir(self, prioridade, id_tarefa, zona_destino):
        """Insere uma nova tarefa na BST mantendo a propriedade de ordenação."""
        novo_no = NoAGV(prioridade, id_tarefa, zona_destino)
        if self.raiz is None:
            self.raiz = novo_no
        else:
            self._inserir_recursivo(self.raiz, novo_no)

    def _inserir_recursivo(self, no_atual, novo_no):
        if novo_no.prioridade < no_atual.prioridade:
            if no_atual.esquerda is None:
                no_atual.esquerda = novo_no
            else:
                self._inserir_recursivo(no_atual.esquerda, novo_no)
        else:
            if no_atual.direita is None:
                no_atual.direita = novo_no
            else:
                self._inserir_recursivo(no_atual.direita, novo_no)

    def buscar_urgente(self):
        """Retorna a tarefa de maior urgência (menor chave de prioridade)."""
        if self.raiz is None:
            return None
        atual = self.raiz
        while atual.esquerda is not None:
            atual = atual.esquerda
        return atual

    # --- MÉTODOS DE TRAVESSIA ---
    def travessia_em_ordem(self, no=None, resultado=None):
        """Retorna as tarefas ordenadas por prioridade crescente (Em-Ordem)."""
        if resultado is None:
            resultado = []
            no = self.raiz
        if no:
            self.travessia_em_ordem(no.esquerda, resultado)
            resultado.append(no)
            self.travessia_em_ordem(no.direita, resultado)
        return resultado

    def travessia_pre_ordem(self, no=None, resultado=None):
        """Utilizado para clonar ou serializar a hierarquia da árvore."""
        if resultado is None:
            resultado = []
            no = self.raiz
        if no:
            resultado.append(no)
            self.travessia_pre_ordem(no.esquerda, resultado)
            self.travessia_pre_ordem(no.direita, resultado)
        return resultado

print("Classe ArvoreDecisaoAGV implementada com sucesso!")

## 3. Simulação Prática: Gerenciamento de Despacho do AGV

In [ ]:
# Inicialização do Gerenciador de Tarefas do AGV
gerenciador_agv = ArvoreDecisaoAGV()

# Cadastrando ordens de transporte enviadas pelo supervisor MES/SCADA
# Formato: (Chave Prioridade, ID Tarefa, Destino)
ordens = [
    (50, "ORD-101", "ALM-201"),       # Transporte Padrão
    (20, "ORD-102", "REATOR-R101"),   # Reagente Crítico (Urgente)
    (80, "ORD-103", "DEP-401"),       # Remoção de Palete Vazio
    (10, "ORD-EMERG", "RECARGA-ST1"), # Bateria Crítica (Urgência Máxima)
    (35, "ORD-104", "AMO-301"),       # Amostragem Laboratório
    (65, "ORD-105", "DOC-101")        # Descarregamento Doca
]

for prio, id_t, dest in ordens:
    gerenciador_agv.inserir(prio, id_t, dest)

print("=== ESCONONOGRAMA DE TAREFAS PROCESSADO VIA BST ===")
tarefas_ordenadas = gerenciador_agv.travessia_em_ordem()
for idx, t in enumerate(tarefas_ordenadas, 1):
    print(f"{idx}º Execução recomendada -> {t}")

no_urgente = gerenciador_agv.buscar_urgente()
print(f"\n🚨 TAREFA DE MAIOR PRIORIDADE IDENTIFICADA: {no_urgente}")

## 4. Análise de Desempenho e Conclusão

1. **Mapeamento $O(\log N)$:** Em uma árvore balanceada, consultar a tarefa mais urgente ou inserir um novo evento demanda no máximo $\approx \log_2 N$ comparações.
2. **Auditoria por Travessia:** A travessia *Em-Ordem* garante a ordenação determinística das tarefas sem necessidade de reordenar arrays na memória heap do controlador do AGV.
3. **Próximos Passos:** Essa estrutura binária serve como base para árvores de busca espacial de pontos no espaço (ex: KD-Trees para nuvens de pontos de LiDAR).